# Reinforcement Learning for Autonomous SRE Control

This notebook demonstrates a Proof of Concept (PoC) for using Reinforcement Learning (RL) with a Large Language Model (LLM) to act as an autonomous Site Reliability Engineering (SRE) controller. The goal is to manage a microservice cluster by making real-time decisions on scaling, traffic routing, and load shedding to maintain service level objectives (SLOs).

**Workflow:**
1.  **Setup & Environment**: Install dependencies, configure the simulation environment (running on Hugging Face Spaces).
2.  **Supervised Fine-tuning (SFT) Data Generation**: Generate a dataset of expert actions using a heuristic policy and augment it with expert examples.
3.  **Supervised Fine-tuning (SFT)**: Train a Qwen2.5-3B-Instruct model with QLoRA using the generated SFT dataset.
4.  **Quality Gate**: Evaluate the SFT model against the heuristic baseline.
5.  **REINFORCE Training**: Further fine-tune the SFT model using the REINFORCE policy gradient algorithm, leveraging rewards from the simulation environment.

In [1]:
# ============================================================
# Cell 1: Install Dependencies
# IMPORTANT: After this cell finishes, RESTART THE RUNTIME
# (Runtime -> Restart session), then continue from Cell 2.
# ============================================================
!pip install -U transformers accelerate bitsandbytes
!pip install unsloth peft trl datasets
!pip install pydantic

print("\n" + "="*60)
print("INSTALLATION COMPLETE — PLEASE RESTART THE RUNTIME NOW")
print("Runtime -> Restart session, then run Cell 2.")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Setup and Environment

This section handles the initial setup, including installing necessary Python packages and configuring connectivity to the simulation environment.

In [1]:
# ============================================================
# Cell 2: Verify Installation, Imports, Upload Project Files
# ============================================================
import importlib, sys, os, json, random, math, re, time
from pathlib import Path
from collections import Counter
from typing import Optional, List, Dict, Any

# Verify critical packages
import torch
import transformers
import peft
import trl

print(f"torch:         {torch.__version__}")
print(f"transformers:  {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"trl:           {trl.__version__}")
print(f"CUDA avail:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GiB")

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Paths
BASE_DIR = Path("/content/AntiAtropos")
OUTPUT_DIR = Path("/colab") # Changed to /colab/ as requested
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nOutput directory: {OUTPUT_DIR}")
print("Setup complete.")

torch:         2.10.0+cu128
transformers:  5.5.0
peft:          0.18.1
trl:           0.24.0
CUDA avail:    True
GPU:           Tesla T4
VRAM total:    15.64 GiB

Output directory: /colab
Setup complete.


### 1.1. Verify Package Installation and Set Global Variables

This cell imports core libraries and verifies their versions, ensuring the environment is correctly set up. It also defines global constants like random seeds and output directories for reproducibility and organized storage.

In [2]:
import requests

HF_SPACE_URL = "https://pranavkk-antiatropos.hf.space"

# Session keeps the TCP connection alive — avoids TLS handshake per request
_session = requests.Session()

def env_reset(task_id="task-1", seed=None):
    payload = {"task_id": task_id}
    if seed is not None:
        payload["seed"] = seed
    resp = _session.post(f"{HF_SPACE_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_step(action_type, target_node_id, parameter):
    payload = {
        "action": {                          # <-- WRAP under "action" key
            "action_type": action_type,
            "target_node_id": target_node_id,
            "parameter": parameter,
        }
    }
    resp = _session.post(f"{HF_SPACE_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Warm up the connection (triggers HF Space cold start if asleep)
test = env_reset("task-1")
print(f"Reset OK — task_id={test.get('observation', {}).get('task_id')}")

Reset OK — task_id=task-1


### 1.2. Configure Hugging Face Space Environment Connectivity

This cell establishes the API connection to the external Hugging Face Space environment, which hosts the microservice cluster simulation. It defines helper functions (`env_reset`, `env_step`) to interact with the environment programmatically. A `requests.Session` is used to maintain persistent connections for efficiency.

In [3]:
# ============================================================
# Cell 4: Verify HF Space connectivity + define helpers
# ============================================================
# No local imports needed — everything comes through the HTTP API.
# We still need these for the heuristic and data generation:
from enum import Enum
from pydantic import BaseModel, Field
import random, json, re, math
from collections import Counter
from pathlib import Path

# Re-define action types (no local models.py needed)
class ActionType(str, Enum):
    NO_OP = "NO_OP"
    SCALE_UP = "SCALE_UP"
    SCALE_DOWN = "SCALE_DOWN"
    REROUTE_TRAFFIC = "REROUTE_TRAFFIC"
    SHED_LOAD = "SHED_LOAD"

VALID_ACTIONS = [a.value for a in ActionType]
VALID_NODES = ["node-0", "node-1", "node-2", "node-3", "node-4"]

# Paths
OUTPUT_DIR = Path("/content/sft_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# Test step
step_result = env_step("NO_OP", "node-0", 0.0)
print(f"Step OK — reward={step_result.get('reward')}, done={step_result.get('done')}")
print("HF Space connectivity verified.")

Step OK — reward=0.615814690097921, done=False
HF Space connectivity verified.


### 1.3. Define Action Types and Verify Environment Interaction

Here, the allowed action types and target nodes for the SRE controller are defined. A quick `env_step` call confirms that the connection to the HF Space environment is active and responsive.

In [4]:
# ============================================================
# Cell 5: Load Model — Qwen2.5-3B-Instruct + QLoRA (attention-only)
# ============================================================
import torch
from unsloth import FastLanguageModel

# ---- Config ----
class ModelConfig:
    model_name: str = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # 3B, fp16-native, 4-bit quantized
    max_seq_length: int = 2048           # T4 safe
    lora_rank: int = 8                   # Smaller for 3B on T4
    lora_alpha: int = 8
    load_in_4bit: bool = True            # QLoRA 4-bit quantization
    dtype = None                         # Auto-detect (fp16 on T4)

cfg = ModelConfig()

print(f"Loading model: {cfg.model_name}")
print(f"LoRA rank={cfg.lora_rank}, alpha={cfg.lora_alpha}, 4-bit={cfg.load_in_4bit}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.model_name,
    max_seq_length=cfg.max_seq_length,
    load_in_4bit=cfg.load_in_4bit,
    dtype=cfg.dtype,
    trust_remote_code=True,
)

# Add LoRA adapters — ATTENTION LAYERS ONLY (saves VRAM, faster on T4)
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention only
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

# Verify VRAM
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM used: {vram_used:.2f} GiB / {vram_total:.2f} GiB")
    print(f"VRAM free: {vram_total - vram_used:.2f} GiB")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nModel: {cfg.model_name}")
print(f"LoRA rank: {cfg.lora_rank}, alpha: {cfg.lora_alpha}")
print(f"Target modules: q_proj, k_proj, v_proj, o_proj (attention only)")
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")
print("Model loaded successfully.")

/tmp/ipykernel_4586/353231185.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
LoRA rank=8, alpha=8, 4-bit=True
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


VRAM used: 2.11 GiB / 15.64 GiB
VRAM free: 13.52 GiB

Model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
LoRA rank: 8, alpha: 8
Target modules: q_proj, k_proj, v_proj, o_proj (attention only)
Trainable params: 3,686,400 (0.22% of 1,702,359,040)
Model loaded successfully.


## 2. Supervised Fine-tuning (SFT) Data Generation

This section focuses on creating a high-quality dataset for supervised fine-tuning. It uses a rule-based heuristic policy to generate diverse scenarios and expert examples to guide the model.

### 2.1. Heuristic Policy and Expert Examples

This cell defines the core heuristic logic that generates actions based on observed cluster states (e.g., latency, queue depth). It also includes a collection of `EXPERT_EXAMPLES` to provide high-quality demonstrations for the LLM, covering critical scenarios and diverse action types. The heuristic is designed to prioritize actions based on cluster health, with internal rotation to ensure a balanced distribution of action types in the dataset.

In [5]:
# ============================================================
# Cell 6: SFT Data Generation — Latency-Based Heuristic + Expert Dataset
#
# DESIGN RATIONALE (from empirical API diagnostic):
#   - queue_depth is ALWAYS 0 after step 1 (M/M/1 drains each tick)
#   - latency_ms is the PRIMARY stress signal (norm 0-1, 1.0 = 1000ms)
#   - node-0/node-4: latency 300-1000ms (97.8% utilization at capacity=3)
#   - node-1/2/3: latency ~38ms (low utilization)
#   - sla_proximity = 1.0 for stressed ingress nodes
#   - FAILED/DEGRADED appear only after tick 20+ in task-2
#   - SCALE_UP capacity 3→4 drops latency from 1000ms to ~63ms
#
# THRESHOLD MAP (normalized → real):
#   latency  0.05 =  50ms  (healthy baseline)
#   latency  0.10 = 100ms  (elevated, proactive scaling)
#   latency  0.30 = 300ms  (stressed)
#   latency  0.50 = 500ms  (critical, near SLA limit)
#   latency  1.00 = 1000ms (SLA violation, capped)
#   sla_prox 0.50 = at 50% of SLA threshold
#   capacity 0.60 = 3 units (default baseline)
# ============================================================

import re, random, textwrap
from collections import Counter
from enum import Enum

VALID_ACTIONS = ["NO_OP", "SCALE_UP", "SCALE_DOWN", "REROUTE_TRAFFIC", "SHED_LOAD"]
VALID_NODES = ["node-0", "node-1", "node-2", "node-3", "node-4"]

# Normalization constants (match environment)
MAX_QUEUE_NORM = 200.0
MAX_LATENCY_NORM = 1000.0
MAX_REQUEST_RATE_NORM = 100.0

# DAG topology
DAG_CHILDREN = {"node-0": ["node-1", "node-2"], "node-2": ["node-3"]}
DAG_CRITICAL_PATH = {"node-0", "node-1", "node-2"}
SAFE_SHED_NODES = {"node-3", "node-4"}  # Non-VIP, off critical path
BASELINE_CAPACITY_NORM = 0.6  # 3 units normalized

# Latency thresholds (normalized)
LATENCY_HEALTHY = 0.05    # 50ms — cluster is healthy
LATENCY_ELEVATED = 0.10   # 100ms — proactive scaling needed
LATENCY_STRESSED = 0.30   # 300ms — aggressive action needed
LATENCY_CRITICAL = 0.50   # 500ms — near SLA violation, urgent
SLA_PROX_THRESHOLD = 0.50 # Node is at 50% of SLA threshold


class ActionType(Enum):
    NO_OP = "NO_OP"
    SCALE_UP = "SCALE_UP"
    SCALE_DOWN = "SCALE_DOWN"
    REROUTE_TRAFFIC = "REROUTE_TRAFFIC"
    SHED_LOAD = "SHED_LOAD"

VALID_ACTIONS = [a.value for a in ActionType]


# ---- System prompt ----
SYSTEM_PROMPT = """You are an autonomous SRE controller managing a five-node microservice cluster.

CLUSTER TOPOLOGY (traffic flows parent -> children):
  node-0 (VIP payment gateway) -> node-1, node-2
  node-2 (catalog) -> node-3 (inventory)
  node-4 (auth, independent ingress)
FAILED nodes have outflow=0 — their children are starved.
Backpressure: overloaded children reduce parent capacity.

ACTIONS (new capacity takes 5 ticks to boot):
  SCALE_UP <node> <amount>   — add capacity (0.3-0.5 normal, 0.6-0.8 heavy surge)
  SCALE_DOWN <node> <amount>  — remove capacity (0.2-0.4 safe, 0.5-0.7 aggressive)
  REROUTE_TRAFFIC <node> <fraction> — move traffic AWAY from this node to healthy peers (0.3-0.7)
  SHED_LOAD <node> <fraction>  — drop incoming traffic (0.3-0.5), NEVER on node-0 (VIP)
  NO_OP — do nothing when cluster is healthy

CRITICAL RULES:
  - node-0 is the VIP payment gateway — NEVER shed its traffic
  - REROUTE_TRAFFIC moves traffic AWAY FROM the target node
  - SCALE_UP clears DEGRADED status on the target node
  - Boot delay is 5 ticks — plan ahead for scaling
  - Use English for reasoning, JSON for the action

ACTION PRIORITY (in order):
  1. REROUTE_TRAFFIC away from FAILED nodes (immediate)
  2. SCALE_UP DEGRADED nodes (urgent)
  3. SCALE_UP nodes with high latency (primary response to M/M/1 stress)
  4. REROUTE_TRAFFIC on high-latency non-VIP nodes (load redistribution)
  5. SHED_LOAD on non-critical nodes only as last resort under severe stress
  6. SCALE_DOWN when cluster is stable and overprovisioned
  7. NO_OP when all nodes have healthy latency

You MUST respond with one sentence of English reasoning, then a JSON action.
The JSON must use EXACTLY these keys: action_type, target_node_id, parameter.
action_type must be one of: SCALE_UP, SCALE_DOWN, REROUTE_TRAFFIC, SHED_LOAD, NO_OP.
target_node_id must be one of: node-0, node-1, node-2, node-3, node-4.
parameter must be a float between 0.0 and 10.0."""

def format_observation(obs_dict, task_id, step, max_steps, reward=0.0, sla_violations=0):
    """Convert API observation dict to natural-language string."""
    lines = [f"Task: {task_id}  Step: {step}/{max_steps}  Reward: {reward:.3f}  SLA violations: {sla_violations}"]
    lines.append("")
    lines.append("Node states:")
    for n in obs_dict.get("nodes", []):
        vip = " (VIP)" if n.get("is_vip") else ""
        status = n.get("status", "HEALTHY")
        q = n.get("queue_depth", 0) * 200
        cap = n.get("capacity", 0) * 5
        pending = n.get("pending_capacity", 0) * 5
        inc = n.get("incoming_request_rate", 0) * 100
        lat = n.get("latency_ms", 0) * 1000
        outflow = n.get("outflow_rate", 0) * 100
        failed = " [FAILED, outflow=0]" if status == "FAILED" else ""
        degraded = " [DEGRADED]" if status == "DEGRADED" else ""
        pending_str = f" (+{pending:.0f} booting)" if pending > 0 else ""
        lines.append(
            f"  {n['node_id']}{vip}: queue={int(q)}, capacity={cap:.0f}{pending_str}, "
            f"incoming={inc:.0f}, latency={lat:.0f}ms, outflow={outflow:.0f}{failed}{degraded}"
        )
    return "\n".join(lines)


def _pick_worst_latency(candidates, node_map, exclude_pending=True):
    """Pick the node with worst latency, preferring DAG-children first (backpressure)."""
    if not candidates:
        return None
    # Sort by latency descending; break ties by queue_depth
    sorted_cands = sorted(candidates, key=lambda n: (n.get("latency_ms", 0), n.get("queue_depth", 0)), reverse=True)
    # If a parent's child also has high latency, prefer the child (break backpressure)
    for n in sorted_cands:
        children = DAG_CHILDREN.get(n["node_id"], [])
        for cid in children:
            child = node_map.get(cid)
            if child and child in candidates:
                if child.get("latency_ms", 0) > n.get("latency_ms", 0) * 0.5:
                    return child
    return sorted_cands[0]


# Internal call counter — guarantees rotation regardless of what
# the calling code passes as `step`.  Incremented on every call.
_call_count = 0


def heuristic_action(obs_dict, task_id, step=0, max_steps=40, episode_reward=0.0):
    """Latency-driven SRE heuristic with INTERNAL rotation for dataset diversity.

    EMPIRICAL BASIS (from API diagnostic):
      - queue_depth is always 0 after initial state (M/M/1 steady-state)
      - latency_ms is the primary stress signal
      - node-0/node-4 (ingress) have latency 300-1000ms at baseline capacity
      - SCALE_UP is the best action for high latency, but REROUTE and SHED
        are also valid responses — rotating produces balanced training data

    IMPORTANT: Uses internal _call_count for rotation, NOT the `step` param.
    Previous versions relied on `step % 3` but the calling code didn't pass
    `step`, so the rotation never fired and produced 100% SCALE_UP.

    Rotation: _call_count % 3 → 0=SCALE_UP, 1=REROUTE, 2=SHED_LOAD
    No pending_capacity check — empirical data shows it's always 0 in obs.
    """
    global _call_count
    _call_count += 1

    nodes = obs_dict.get("nodes", [])
    if not nodes:
        return ActionType.NO_OP, "node-0", 0.0

    node_map = {n["node_id"]: n for n in nodes}
    avg_latency = sum(n.get("latency_ms", 0) for n in nodes) / len(nodes)
    failed_nodes = [n for n in nodes if n.get("status") == "FAILED"]
    degraded_nodes = [n for n in nodes if n.get("status") == "DEGRADED"]

    # ── PRIORITY 1: REROUTE away from FAILED nodes (immediate) ──
    if failed_nodes:
        target = failed_nodes[0]
        return ActionType.REROUTE_TRAFFIC, target["node_id"], 0.7

    # ── PRIORITY 2: SCALE_UP DEGRADED nodes (urgent) ──
    if degraded_nodes:
        target = degraded_nodes[0]
        q = target.get("queue_depth", 0) * MAX_QUEUE_NORM
        param = 0.7 if q > 60 else 0.5
        return ActionType.SCALE_UP, target["node_id"], param

    # ── PRIORITY 3: Latency-based actions with INTERNAL ROTATION ──
    # NOTE: No pending_capacity check — empirical data shows it's always
    # reported as 0 in observations even after SCALE_UP.  The rotation
    # itself prevents SCALE_UP spam by interleaving REROUTE/SHED.
    high_latency = [
        n for n in nodes
        if n.get("latency_ms", 0) > LATENCY_ELEVATED
        and n.get("status") != "FAILED"
    ]
    vip_high = [n for n in high_latency if n.get("is_vip", False)]
    non_vip_high = [n for n in high_latency if not n.get("is_vip", False)]
    shed_candidates = [n for n in high_latency if n["node_id"] in SAFE_SHED_NODES]

    if high_latency:
        strategy = _call_count % 3  # INTERNAL counter — always rotates

        if strategy == 0:
            # ── SCALE_UP: add capacity ──
            # Alternate target: even calls → VIP (node-0), odd → non-VIP (node-4)
            if _call_count % 2 == 0 and vip_high:
                target = max(vip_high, key=lambda n: n.get("latency_ms", 0))
            elif non_vip_high:
                target = max(non_vip_high, key=lambda n: n.get("latency_ms", 0))
            elif vip_high:
                target = max(vip_high, key=lambda n: n.get("latency_ms", 0))
            else:
                target = max(high_latency, key=lambda n: n.get("latency_ms", 0))
            lat = target.get("latency_ms", 0)
            param = 0.5 if lat > LATENCY_STRESSED else 0.3
            return ActionType.SCALE_UP, target["node_id"], param

        elif strategy == 1:
            # ── REROUTE_TRAFFIC: redistribute load from non-VIP nodes ──
            if non_vip_high:
                target = max(non_vip_high, key=lambda n: n.get("latency_ms", 0))
                lat = target.get("latency_ms", 0)
                param = 0.5 if lat > LATENCY_CRITICAL else 0.3
                return ActionType.REROUTE_TRAFFIC, target["node_id"], param
            # No non-VIP high-latency node — fallback to SCALE_UP on VIP
            if vip_high:
                target = max(vip_high, key=lambda n: n.get("latency_ms", 0))
                return ActionType.SCALE_UP, target["node_id"], 0.5
            target = max(high_latency, key=lambda n: n.get("latency_ms", 0))
            return ActionType.SCALE_UP, target["node_id"], 0.3

        else:  # strategy == 2
            # ── SHED_LOAD: drop non-critical traffic ──
            if shed_candidates:
                target = max(shed_candidates, key=lambda n: n.get("latency_ms", 0))
                return ActionType.SHED_LOAD, target["node_id"], 0.3
            # No shed candidates — fallback to REROUTE on non-VIP
            if non_vip_high:
                target = max(non_vip_high, key=lambda n: n.get("latency_ms", 0))
                return ActionType.REROUTE_TRAFFIC, target["node_id"], 0.3
            # Last fallback: SCALE_UP
            target = max(high_latency, key=lambda n: n.get("latency_ms", 0))
            return ActionType.SCALE_UP, target["node_id"], 0.3

    # ── PRIORITY 4: SCALE_DOWN overprovisioned nodes when cluster is quiet ──
    if avg_latency < LATENCY_HEALTHY:
        non_vips = [
            n for n in nodes
            if not n.get("is_vip", False)
            and n.get("status") != "FAILED"
            and n.get("capacity", 0) > BASELINE_CAPACITY_NORM + 0.2
        ]
        if non_vips:
            target = max(non_vips, key=lambda n: n.get("capacity", 0))
            excess = target.get("capacity", 0) - BASELINE_CAPACITY_NORM
            param = min(0.4, excess * 0.5)
            return ActionType.SCALE_DOWN, target["node_id"], param

    # ── PRIORITY 5: NO_OP when all nodes have healthy latency ──
    return ActionType.NO_OP, "node-0", 0.0


def generate_reasoning(action_type, target_node_id, parameter, obs_dict, task_id):
    """Generate an English reasoning sentence for the chosen action."""
    nodes = obs_dict.get("nodes", [])
    node_map = {n["node_id"]: n for n in nodes}
    target = node_map.get(target_node_id, {})
    lat = target.get("latency_ms", 0) * MAX_LATENCY_NORM
    status = target.get("status", "HEALTHY")
    q = target.get("queue_depth", 0) * MAX_QUEUE_NORM
    children = DAG_CHILDREN.get(target_node_id, [])

    if action_type == ActionType.SCALE_UP:
        if status == "DEGRADED":
            return f"Node {target_node_id} is DEGRADED with latency={int(lat)}ms, scaling up to restore service and clear degraded status."
        if lat > 500:
            return f"Node {target_node_id} has critical latency={int(lat)}ms, scaling up urgently to prevent SLA violation."
        if children:
            children_stressed = any(
                node_map.get(c, {}).get("latency_ms", 0) > LATENCY_ELEVATED for c in children
            )
            if children_stressed:
                return f"Backpressure from children of {target_node_id} is causing latency={int(lat)}ms, scaling up to break the pressure chain."
            return f"Node {target_node_id} latency={int(lat)}ms is elevated, proactively scaling up to stay ahead of demand."
        return f"Node {target_node_id} latency={int(lat)}ms is above threshold, scaling up to reduce response time."

    elif action_type == ActionType.SCALE_DOWN:
        return f"Cluster is stable with low latency, scaling down {target_node_id} to reduce infrastructure cost."

    elif action_type == ActionType.REROUTE_TRAFFIC:
        if status == "FAILED":
            return f"Node {target_node_id} has FAILED with outflow=0, rerouting its traffic to healthy peers to prevent request loss."
        return f"Node {target_node_id} has latency={int(lat)}ms, rerouting traffic to healthy peers to reduce pressure."

    elif action_type == ActionType.SHED_LOAD:
        return f"Cluster is under severe stress, shedding {parameter:.0%} of traffic on non-critical node {target_node_id} to protect the critical DAG path."

    elif action_type == ActionType.NO_OP:
        return "All nodes have healthy latency, no intervention needed."

    return "Taking action based on current cluster state."


def make_assistant_text(action_type, target_node_id, parameter):
    """Build the assistant response: reasoning + JSON action."""
    if isinstance(action_type, ActionType):
        action_str = action_type.value
    else:
        action_str = str(action_type)
    return f'{action_str} {target_node_id} {parameter:.1f} {{"action_type": "{action_str}", "target_node_id": "{target_node_id}", "parameter": {parameter}}}'


# ---- Expert augmentation examples ----
EXPERT_EXAMPLES = []

# Expert: REROUTE_TRAFFIC on FAILED node (task-2 scenario, after tick 20)
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-2  Step: 25/60  Reward: 0.320  SLA violations: 3

Node states:
  node-0 (VIP): queue=0, capacity=3, incoming=44, latency=350ms, outflow=44
  node-1: queue=0, capacity=0, incoming=22, latency=0ms, outflow=0 [FAILED, outflow=0]
  node-2: queue=0, capacity=3, incoming=22, latency=38ms, outflow=22
  node-3: queue=0, capacity=3, incoming=11, latency=38ms, outflow=11
  node-4: queue=0, capacity=3, incoming=44, latency=350ms, outflow=44""",
    "assistant": 'Node node-1 has FAILED with outflow=0, rerouting its traffic to healthy peers to prevent request loss. {"action_type": "REROUTE_TRAFFIC", "target_node_id": "node-1", "parameter": 0.7}',
})

# Expert: SCALE_UP after REROUTE to handle absorbed traffic
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-2  Step: 28/60  Reward: 0.290  SLA violations: 4

Node states:
  node-0 (VIP): queue=0, capacity=3, incoming=44, latency=350ms, outflow=44
  node-1: queue=0, capacity=0, incoming=22, latency=0ms, outflow=0 [FAILED, outflow=0]
  node-2: queue=0, capacity=3, incoming=32, latency=65ms, outflow=32
  node-3: queue=0, capacity=3, incoming=16, latency=38ms, outflow=16
  node-4: queue=0, capacity=3, incoming=44, latency=350ms, outflow=44""",
    "assistant": 'Node-2 latency=65ms is rising from absorbing rerouted traffic from failed node-1, scaling up to handle the increased load. {"action_type": "SCALE_UP", "target_node_id": "node-2", "parameter": 0.5}',
})

# Expert: SCALE_UP on high-latency ingress node (most common scenario)
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 5/40  Reward: 0.610  SLA violations: 1

Node states:
  node-0 (VIP): queue=0, capacity=3, incoming=44, latency=829ms, outflow=44
  node-1: queue=0, capacity=3, incoming=22, latency=39ms, outflow=22
  node-2: queue=0, capacity=3, incoming=22, latency=39ms, outflow=22
  node-3: queue=0, capacity=3, incoming=22, latency=39ms, outflow=22
  node-4: queue=0, capacity=3, incoming=44, latency=829ms, outflow=44""",
    "assistant": 'Node node-0 has critical latency=829ms, scaling up urgently to prevent SLA violation. {"action_type": "SCALE_UP", "target_node_id": "node-0", "parameter": 0.7}',
})

# Expert: SCALE_UP node-4 after node-0 is handled
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 8/40  Reward: 0.620  SLA violations: 1

Node states:
  node-0 (VIP): queue=0, capacity=4 (+3 booting), incoming=44, latency=350ms, outflow=44
  node-1: queue=0, capacity=3, incoming=22, latency=39ms, outflow=22
  node-2: queue=0, capacity=3, incoming=22, latency=39ms, outflow=22
  node-3: queue=0, capacity=3, incoming=22, latency=39ms, outflow=22
  node-4: queue=0, capacity=3, incoming=44, latency=900ms, outflow=44""",
    "assistant": 'Node node-4 has critical latency=900ms, scaling up to reduce response time. {"action_type": "SCALE_UP", "target_node_id": "node-4", "parameter": 0.7}',
})

# Expert: REROUTE_TRAFFIC on high-latency node-4 (non-VIP)
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 12/40  Reward: 0.550  SLA violations: 2

Node states:
  node-0 (VIP): queue=0, capacity=4, incoming=44, latency=63ms, outflow=44
  node-1: queue=0, capacity=3, incoming=22, latency=38ms, outflow=22
  node-2: queue=0, capacity=3, incoming=22, latency=38ms, outflow=22
  node-3: queue=0, capacity=3, incoming=22, latency=38ms, outflow=22
  node-4: queue=0, capacity=3, incoming=44, latency=875ms, outflow=44""",
    "assistant": 'Node node-4 has latency=875ms, rerouting traffic to healthy peers to reduce pressure. {"action_type": "REROUTE_TRAFFIC", "target_node_id": "node-4", "parameter": 0.5}',
})

# Expert: SHED_LOAD on node-4 under severe stress
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-3  Step: 25/60  Reward: 0.250  SLA violations: 5

Node states:
  node-0 (VIP): queue=0, capacity=4, incoming=44, latency=63ms, outflow=44
  node-1: queue=0, capacity=3, incoming=22, latency=38ms, outflow=22
  node-2: queue=0, capacity=5, incoming=22, latency=30ms, outflow=22
  node-3: queue=0, capacity=3, incoming=22, latency=38ms, outflow=22
  node-4: queue=0, capacity=3, incoming=44, latency=1000ms, outflow=44""",
    "assistant": 'Cluster is under severe stress, shedding 30% of traffic on non-critical node node-4 to protect the critical DAG path. {"action_type": "SHED_LOAD", "target_node_id": "node-4", "parameter": 0.3}',
})

# Expert: SHED_LOAD on node-3 (leaf, non-critical) under surge
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-3  Step: 30/60  Reward: 0.200  SLA violations: 6

Node states:
  node-0 (VIP): queue=0, capacity=4, incoming=44, latency=63ms, outflow=44
  node-1: queue=0, capacity=5, incoming=80, latency=42ms, outflow=75
  node-2: queue=0, capacity=5, incoming=80, latency=42ms, outflow=75
  node-3: queue=0, capacity=3, incoming=40, latency=55ms, outflow=40
  node-4: queue=0, capacity=3, incoming=44, latency=1000ms, outflow=44""",
    "assistant": 'Cluster is under severe stress, shedding 30% of traffic on non-critical node node-3 to protect the critical DAG path. {"action_type": "SHED_LOAD", "target_node_id": "node-3", "parameter": 0.3}',
})

# Expert: SCALE_DOWN on idle node after scaling
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 35/40  Reward: 0.850  SLA violations: 0

Node states:
  node-0 (VIP): queue=0, capacity=4, incoming=44, latency=32ms, outflow=44
  node-1: queue=0, capacity=3, incoming=22, latency=20ms, outflow=22
  node-2: queue=0, capacity=4, incoming=22, latency=25ms, outflow=22
  node-3: queue=0, capacity=3, incoming=11, latency=18ms, outflow=11
  node-4: queue=0, capacity=4, incoming=44, latency=32ms, outflow=44""",
    "assistant": 'Cluster is stable with low latency, scaling down node-2 to reduce infrastructure cost. {"action_type": "SCALE_DOWN", "target_node_id": "node-2", "parameter": 0.3}',
})

# Expert: NO_OP on healthy cluster
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 20/40  Reward: 0.900  SLA violations: 0

Node states:
  node-0 (VIP): queue=0, capacity=4, incoming=44, latency=40ms, outflow=44
  node-1: queue=0, capacity=3, incoming=22, latency=22ms, outflow=22
  node-2: queue=0, capacity=3, incoming=22, latency=22ms, outflow=22
  node-3: queue=0, capacity=3, incoming=11, latency=18ms, outflow=11
  node-4: queue=0, capacity=4, incoming=44, latency=40ms, outflow=44""",
    "assistant": 'All nodes have healthy latency, no intervention needed. {"action_type": "NO_OP", "target_node_id": "node-0", "parameter": 0.0}',
})

print("Latency-based heuristic and expert examples defined.")
print(f"Expert examples: {len(EXPERT_EXAMPLES)}")
expert_actions = set()
for e in EXPERT_EXAMPLES:
    match = re.search(r'"action_type":\s*"(\w+)"', e["assistant"])
    if match:
        expert_actions.add(match.group(1))
print(f"Action types in experts: {expert_actions}")
print(f"\nKey design: SCALE_UP triggers on latency > {LATENCY_ELEVATED} (100ms), not queue_depth")
print(f"Action rotation: _call_count%3 → 0=SCALE_UP, 1=REROUTE, 2=SHED_LOAD")
print(f"NO pending_capacity check — empirical data shows it's always 0 in observations")
print(f"Target rotation: even calls → VIP (node-0), odd → non-VIP (node-4)")
print(f"Expected: ~33% SCALE_UP, ~25% REROUTE, ~25% SHED, ~12% NO_OP, ~5% SCALE_DOWN")

Latency-based heuristic and expert examples defined.
Expert examples: 9
Action types in experts: {'SCALE_DOWN', 'NO_OP', 'SCALE_UP', 'REROUTE_TRAFFIC', 'SHED_LOAD'}

Key design: SCALE_UP triggers on latency > 0.1 (100ms), not queue_depth
Action rotation: _call_count%3 → 0=SCALE_UP, 1=REROUTE, 2=SHED_LOAD
NO pending_capacity check — empirical data shows it's always 0 in observations
Target rotation: even calls → VIP (node-0), odd → non-VIP (node-4)
Expected: ~33% SCALE_UP, ~25% REROUTE, ~25% SHED, ~12% NO_OP, ~5% SCALE_DOWN


### 2.2. Run Episodes and Build SFT Dataset

This cell executes multiple episodes across different tasks using the defined heuristic policy. It collects observation-action pairs, formats them into a chat-like structure, and augments them with the `EXPERT_EXAMPLES`. The generated data is then split into training, validation, and test sets, and saved as JSONL files, ready for supervised fine-tuning.

In [6]:
# ============================================================
# Cell 7: Run Episodes & Build SFT Dataset
# ============================================================

NUM_EPISODES_PER_TASK = 8
MAX_STEPS = 40
NO_OP_CAP = 0.20

all_examples = []
action_counts = Counter()
node_counts = Counter()
reward_sum = 0.0
reward_count = 0

for task_id in ["task-1", "task-2", "task-3"]:
    print(f"\n{'='*60}")
    print(f"Generating episodes for {task_id}")
    print(f"{'='*60}")
    task_rewards = []

    for ep_idx in range(NUM_EPISODES_PER_TASK):
        seed = SEED + ep_idx * 100 + hash(task_id) % 1000

        reset_resp = env_reset(task_id=task_id, seed=seed)
        obs_dict = reset_resp.get("observation", reset_resp)
        sla_violations = obs_dict.get("sla_violations", 0)
        episode_reward = 0.0
        ep_actions = []

        for step in range(1, MAX_STEPS + 1):
            obs_text = format_observation(obs_dict, task_id, step, MAX_STEPS, episode_reward, sla_violations)
            action_type, target_node_id, parameter = heuristic_action(obs_dict, task_id)
            reasoning = generate_reasoning(action_type, target_node_id, parameter, obs_dict, task_id)

            action_json = json.dumps({
                "action_type": action_type.value,
                "target_node_id": target_node_id,
                "parameter": round(float(parameter), 4),
            })
            assistant_text = f"{reasoning} {action_json}"

            all_examples.append({
                "system": SYSTEM_PROMPT,
                "user": obs_text,
                "assistant": assistant_text,
                "task_id": task_id,
                "action_type": action_type.value,
                "target_node_id": target_node_id,
                "reward": episode_reward,
            })

            action_counts[action_type.value] += 1
            node_counts[target_node_id] += 1
            ep_actions.append(action_type.value)

            step_resp = env_step(action_type.value, target_node_id, parameter)
            obs_dict = step_resp.get("observation", step_resp)
            episode_reward = step_resp.get("reward", episode_reward)
            reward_sum += episode_reward
            reward_count += 1
            sla_violations = obs_dict.get("sla_violations", sla_violations)

            if step_resp.get("done", False):
                break

        task_rewards.append(episode_reward)
        action_dist = Counter(ep_actions)
        print(f"  Ep {ep_idx+1}/{NUM_EPISODES_PER_TASK} | reward={episode_reward:.3f} | "
              f"actions: {dict(action_dist)}")

    avg_r = sum(task_rewards) / len(task_rewards)
    print(f"  >> {task_id} avg reward: {avg_r:.3f}")

print(f"\nHeuristic episodes generated: {len(all_examples)} examples")
print(f"Action distribution: {dict(action_counts)}")

# ---- Add expert examples ----
for ex in EXPERT_EXAMPLES:
    at_match = re.search(r'"action_type":\s*"(\w+)"', ex["assistant"])
    tn_match = re.search(r'"target_node_id":\s*"(node-\d+)"', ex["assistant"])
    if at_match and tn_match:
        all_examples.append({
            "system": ex["system"],
            "user": ex["user"],
            "assistant": ex["assistant"],
            "task_id": "expert",
            "action_type": at_match.group(1),
            "target_node_id": tn_match.group(1),
            "reward": 0.0,
        })
        action_counts[at_match.group(1)] += 1
        node_counts[tn_match.group(1)] += 1

print(f"After expert augmentation: {len(all_examples)} examples")

# ---- Filter: cap NO_OP at 40% ----
noop_examples = [e for e in all_examples if e["action_type"] == "NO_OP"]
non_noop_examples = [e for e in all_examples if e["action_type"] != "NO_OP"]
max_noop = int(len(all_examples) * NO_OP_CAP)
if len(noop_examples) > max_noop:
    random.shuffle(noop_examples)
    noop_examples = noop_examples[:max_noop]
    all_examples = non_noop_examples + noop_examples
    print(f"NO_OP capped: keeping {max_noop} of {len(noop_examples)} NO_OP examples")

# ---- Deduplicate ----
seen = set()
unique_examples = []
for e in all_examples:
    key = (e["user"][:100], e["action_type"], e["target_node_id"])
    if key not in seen:
        seen.add(key)
        unique_examples.append(e)
all_examples = unique_examples
print(f"After dedup: {len(all_examples)} examples")

# ---- Shuffle ----
random.shuffle(all_examples)

# ---- Train/Test/Val split (70/15/15) ----
from sklearn.model_selection import train_test_split

train_ex, temp_ex = train_test_split(
    all_examples, test_size=0.30, random_state=SEED,
)
test_ex, val_ex = train_test_split(
    temp_ex, test_size=0.50, random_state=SEED,
)

print(f"\nDataset split:")
print(f"  Train: {len(train_ex)}")
print(f"  Test:  {len(test_ex)}")
print(f"  Val:   {len(val_ex)}")

# ---- Save as JSONL ----
def save_jsonl(examples, path):
    with open(path, "w") as f:
        for ex in examples:
            record = {
                "messages": [
                    {"role": "system", "content": ex["system"]},
                    {"role": "user", "content": ex["user"]},
                    {"role": "assistant", "content": ex["assistant"]},
                ],
                "task_id": ex.get("task_id", ""),
                "action_type": ex.get("action_type", ""),
                "target_node_id": ex.get("target_node_id", ""),
            }
            f.write(json.dumps(record) + "\n")

save_jsonl(train_ex, OUTPUT_DIR / "sft_train.jsonl")
save_jsonl(test_ex, OUTPUT_DIR / "sft_test.jsonl")
save_jsonl(val_ex, OUTPUT_DIR / "sft_val.jsonl")

# ---- Print statistics ----
final_action_counts = Counter(e["action_type"] for e in all_examples)
final_node_counts = Counter(e["target_node_id"] for e in all_examples)
total = len(all_examples)

print(f"\n{'='*60}")
print(f"DATASET STATISTICS ({total} total examples)")
print(f"{'='*60}")
print(f"\nAction type distribution:")
for at in VALID_ACTIONS:
    cnt = final_action_counts.get(at, 0)
    pct = 100 * cnt / total if total > 0 else 0
    flag = " <-- UNDERREPRESENTED" if pct < 3 and at != "NO_OP" else ""
    print(f"  {at:20s}: {cnt:5d} ({pct:5.1f}%){flag}")

print(f"\nTarget node distribution:")
for nid in VALID_NODES:
    cnt = final_node_counts.get(nid, 0)
    pct = 100 * cnt / total if total > 0 else 0
    flag = " <-- UNDERREPRESENTED" if pct < 5 else ""
    print(f"  {nid:10s}: {cnt:5d} ({pct:5.1f}%){flag}")

print(f"\nAverage reward: {reward_sum / reward_count:.4f}" if reward_count > 0 else "\nNo rewards computed")
print(f"\nFiles saved to {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.glob("sft_*.jsonl")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")


Generating episodes for task-1
  Ep 1/8 | reward=0.558 | actions: {'REROUTE_TRAFFIC': 14, 'SHED_LOAD': 13, 'SCALE_UP': 13}
  Ep 2/8 | reward=0.558 | actions: {'SHED_LOAD': 14, 'SCALE_UP': 13, 'REROUTE_TRAFFIC': 13}
  Ep 3/8 | reward=0.558 | actions: {'SCALE_UP': 17, 'REROUTE_TRAFFIC': 12, 'SHED_LOAD': 10, 'NO_OP': 1}
  Ep 4/8 | reward=0.795 | actions: {'REROUTE_TRAFFIC': 11, 'SHED_LOAD': 13, 'SCALE_UP': 16}
  Ep 5/8 | reward=0.561 | actions: {'SHED_LOAD': 12, 'SCALE_UP': 16, 'REROUTE_TRAFFIC': 12}
  Ep 6/8 | reward=0.558 | actions: {'SCALE_UP': 16, 'REROUTE_TRAFFIC': 12, 'SHED_LOAD': 12}
  Ep 7/8 | reward=0.558 | actions: {'REROUTE_TRAFFIC': 14, 'SHED_LOAD': 13, 'SCALE_UP': 13}
  Ep 8/8 | reward=0.558 | actions: {'SHED_LOAD': 12, 'SCALE_UP': 16, 'REROUTE_TRAFFIC': 12}
  >> task-1 avg reward: 0.588

Generating episodes for task-2
  Ep 1/8 | reward=0.566 | actions: {'SCALE_UP': 15, 'REROUTE_TRAFFIC': 12, 'SHED_LOAD': 13}
  Ep 2/8 | reward=0.558 | actions: {'REROUTE_TRAFFIC': 13, 'SHED_L

## 3. Supervised Fine-tuning (SFT) with QLoRA

This section covers the supervised fine-tuning of the pre-trained Qwen2.5-3B-Instruct model using the dataset generated in the previous step. QLoRA is used for efficient memory usage.

In [7]:
# ============================================================
# Cell 8: SFT Training with Unsloth + SFTTrainer (QLoRA)
# ============================================================
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_jsonl(OUTPUT_DIR / "sft_train.jsonl")
val_records = load_jsonl(OUTPUT_DIR / "sft_val.jsonl")

# Pre-format into a "text" field using Qwen2.5 chatml manually
def apply_template(record):
    parts = []
    for msg in record["messages"]:
        role = msg["role"]
        content = msg["content"]
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    # Add assistant generation prompt
    parts.append("<|im_start|>assistant\n")
    return {"text": "\n".join(parts)}

train_dataset = Dataset.from_list(train_records).map(apply_template)
val_dataset = Dataset.from_list(val_records).map(apply_template)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Sample text (first 300 chars): {train_dataset[0]['text'][:300]}")

# ---- Config ----
class TC:
    per_device_train_batch_size = 4    # Increased for better GPU utilization
    gradient_accumulation_steps = 8
    learning_rate = 2e-4
    num_train_epochs = 2               # Increased due to faster training
    max_seq_length = 2048
    warmup_steps = 20
    weight_decay = 0.01
    logging_steps = 7                  # Adjusted for more frequent logging
    save_steps = 100                   # Adjusted for faster training
    eval_strategy = "steps"
    eval_steps = 7                     # Adjusted for more frequent evaluation
    save_total_limit = 3
    bf16 = False                       # Disabled bfloat16 as requested
    optim = "adamw_8bit"
    output_dir = str(OUTPUT_DIR / "sft_lora_adapter")

cfg = TC()

print(f"\nSFT Config:")
print(f"  epochs={cfg.num_train_epochs}, lr={cfg.learning_rate}")
print(f"  batch={cfg.per_device_train_batch_size}, grad_accum={cfg.gradient_accumulation_steps}")
print(f"  effective_batch={cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}")
print(f"  logging every {cfg.logging_steps} steps, eval every {cfg.eval_steps} steps")

# ---- Trainer ----
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        num_train_epochs=cfg.num_train_epochs,
        max_seq_length=cfg.max_seq_length,
        warmup_steps=cfg.warmup_steps,
        weight_decay=cfg.weight_decay,
        logging_steps=cfg.logging_steps,
        save_steps=cfg.save_steps,
        eval_strategy=cfg.eval_strategy,
        eval_steps=cfg.eval_steps,
        save_total_limit=cfg.save_total_limit,
        bf16=cfg.bf16,
        optim=cfg.optim,
        output_dir=cfg.output_dir,
        dataset_text_field="text",
        packing=False,
    ),
)

print("\nStarting SFT training...")
train_result = trainer.train()

print(f"\nSFT Training Complete!")
print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")

trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"  Adapter saved to {cfg.output_dir}")

# eval_result = trainer.evaluate() # Evaluation is already handled during training due to eval_strategy="steps"
# print(f"  Eval loss: {eval_result.get('eval_loss', 'N/A')}")

if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM: {vram_used:.2f}/{vram_total:.2f} GiB")

Map:   0%|          | 0/582 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Training samples: 582
Validation samples: 125
Sample text (first 300 chars): <|im_start|>system
You are an autonomous SRE controller managing a five-node microservice cluster.

CLUSTER TOPOLOGY (traffic flows parent -> children):
  node-0 (VIP payment gateway) -> node-1, node-2
  node-2 (catalog) -> node-3 (inventory)
  node-4 (auth, independent ingress)
FAILED nodes have ou

SFT Config:
  epochs=2, lr=0.0002
  batch=4, grad_accum=8
  effective_batch=32
  logging every 7 steps, eval every 7 steps


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/582 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/125 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss
7,2.182384,2.129317
14,2.062419,1.938685
21,1.831849,1.671376
28,1.563358,1.416052
35,1.340591,1.251937
38,1.340591,1.227380


Unsloth: Restored added_tokens_decoder metadata in /content/sft_output/sft_lora_adapter/checkpoint-38/tokenizer_config.json.



SFT Training Complete!
  Train loss: 1.7524
  Total steps: 38


Unsloth: Restored added_tokens_decoder metadata in /content/sft_output/sft_lora_adapter/tokenizer_config.json.


  Adapter saved to /content/sft_output/sft_lora_adapter
  VRAM: 2.15/15.64 GiB


## 4. Lightweight Quality Gate

Before proceeding to Reinforcement Learning, a quality gate evaluates the SFT model's performance against the heuristic baseline. This ensures the SFT model has learned reasonable policies and can generate valid actions, preventing further training on a potentially underperforming base model.

In [8]:
# ============================================================
# Cell 9: Lightweight Quality Gate — Live Simulated Episodes
# ============================================================
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")
warnings.filterwarnings("ignore", message=".*The attention mask API under `transformers.modeling_attn_mask_utils`.*")

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

# Fix Unsloth 4-bit generate() bug: move_to_device gets None device
import unsloth.models._utils as _unsloth_utils
import torch
_orig_move = _unsloth_utils.move_to_device
def _patched_move(target_device, *tensors):
    if target_device is None:
        target_device = torch.device("cuda:0")
    return _orig_move(target_device, *tensors)
_unsloth_utils.move_to_device = _patched_move

FastLanguageModel.for_inference(model)
model.config.max_length = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Patch Unsloth 4-bit model.device = None issue
if model.device is None or str(model.device) == "None":
    model.hf_device_map = {"": 0}
    model.device = torch.device("cuda:0")

print(f"Inference device: {DEVICE}")


def parse_model_action(text):
    """Extract action from model output text."""
    try:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end < start:
            return None, "no JSON found"
        obj = json.loads(text[start:end+1])
        at = str(obj.get("action_type", "")).upper()
        if at not in VALID_ACTIONS:
            return None, f"invalid action_type: {at}"
        nid = str(obj.get("target_node_id", ""))
        if nid not in VALID_NODES:
            return None, f"invalid target_node_id: {nid}"
        param = float(obj.get("parameter", 0.0))
        return (at, nid, param), "ok"
    except Exception as e:
        return None, str(e)


def build_inference_prompt(obs_dict, task_id, step, max_steps, reward=0.0, sla_violations=0):
    """Build the user prompt for inference (matches training format)."""
    return format_observation(obs_dict, task_id, step, max_steps, reward, sla_violations)


def run_episode_with_model(task_id, max_steps, model, tokenizer):
    """Run one episode using the SFT model via HF Space."""
    reset_resp = env_reset(task_id=task_id)
    obs_dict = reset_resp.get("observation", reset_resp)
    rewards = []
    crashes = 0
    invalid_actions = 0
    action_log = []

    for step in range(1, max_steps + 1):
        obs_text = build_inference_prompt(
            obs_dict, task_id, step, max_steps,
            rewards[-1] if rewards else 0.0,
            sum(1 for r in rewards if r < 0.3)
        )

        # Build chatml prompt (matches training format)
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{obs_text}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        parsed, err = parse_model_action(generated)
        if parsed is None:
            crashes += 1
            at, nid, param = "NO_OP", "node-0", 0.0
            action_log.append(f"step={step} INVALID ({err})")
        else:
            at, nid, param = parsed
            action_log.append(f"step={step} {at} {nid} p={param:.2f}")

        # Step via HF Space
        step_resp = env_step(at, nid, param)
        obs_dict = step_resp.get("observation", step_resp)
        reward = step_resp.get("reward", 0.0)
        rewards.append(reward)

        if step_resp.get("done", False):
            break

    return {
        "rewards": rewards,
        "avg_reward": sum(rewards) / len(rewards) if rewards else 0.0,
        "crashes": crashes,
        "action_log": action_log,
    }


def run_episode_with_heuristic(task_id, max_steps):
    """Run one episode using the heuristic baseline via HF Space."""
    reset_resp = env_reset(task_id=task_id)
    obs_dict = reset_resp.get("observation", reset_resp)
    rewards = []

    for step in range(1, max_steps + 1):
        at, nid, param = heuristic_action(obs_dict, task_id)
        step_resp = env_step(at.value, nid, param)
        obs_dict = step_resp.get("observation", step_resp)
        reward = step_resp.get("reward", 0.0)
        rewards.append(reward)

        if step_resp.get("done", False):
            break

    return {
        "rewards": rewards,
        "avg_reward": sum(rewards) / len(rewards) if rewards else 0.0,
    }


# ---- Run evaluation ----
EVAL_EPISODES = 3
EVAL_STEPS = 40

print(f"Running {EVAL_EPISODES} episodes per task ({EVAL_STEPS} steps each)...")
print(f"{'='*70}")

results = {}
for task_id in ["task-1", "task-2", "task-3"]:
    sft_rewards = []
    heuristic_rewards = []
    total_crashes = 0

    for ep in range(EVAL_EPISODES):
        print(f"  {task_id} ep {ep+1}/{EVAL_EPISODES} (SFT)...")
        sft_result \
        = run_episode_with_model(task_id, EVAL_STEPS, model, tokenizer)
        sft_rewards.append(sft_result["avg_reward"])
        total_crashes += sft_result["crashes"]

        print(f"  {task_id} ep {ep+1}/{EVAL_EPISODES} (heuristic)...")
        heur_result = run_episode_with_heuristic(task_id, EVAL_STEPS)
        heuristic_rewards.append(heur_result["avg_reward"])

    sft_avg = sum(sft_rewards) / len(sft_rewards)
    heur_avg = sum(heuristic_rewards) / len(heuristic_rewards)

    results[task_id] = {
        "sft_avg_reward": sft_avg,
        "heuristic_avg_reward": heur_avg,
        "crashes": total_crashes,
        "sft_beats_heuristic": sft_avg >= heur_avg,
    }

    status = "SFT WINS" if sft_avg >= heur_avg else "HEURISTIC WINS"
    print(f"\n{task_id}:")
    print(f"  SFT avg reward:       {sft_avg:.4f}")
    print(f"  Heuristic avg reward: {heur_avg:.4f}")
    print(f"  Result: {status}")
    print(f"  Crashes: {total_crashes}")

# ---- Summary ----
tasks_won = sum(1 for r in results.values() if r["sft_beats_heuristic"])
total_crashes = sum(r["crashes"] for r in results.values())

print(f"\n{'='*70}")
print(f"QUALITY GATE SUMMARY")
print(f"{'='*70}")
print(f"SFT beats heuristic on: {tasks_won}/3 tasks")
print(f"Total crashes: {total_crashes}")
print(f"\nGate results:")
gate_pass = total_crashes == 0 and tasks_won >= 2
print(f"  Zero crashes: {'PASS' if total_crashes == 0 else 'FAIL'}")
print(f"  Beat heuristic >= 2/3: {'PASS' if tasks_won >= 2 else 'FAIL'}")
print(f"\nOverall: {'GATE PASSED' if gate_pass else 'GATE FAILED — consider more SFT epochs or larger dataset'}")

Inference device: cuda
Running 3 episodes per task (40 steps each)...
  task-1 ep 1/3 (SFT)...
  task-1 ep 1/3 (heuristic)...
  task-1 ep 2/3 (SFT)...
  task-1 ep 2/3 (heuristic)...
  task-1 ep 3/3 (SFT)...
  task-1 ep 3/3 (heuristic)...

task-1:
  SFT avg reward:       0.6333
  Heuristic avg reward: 0.6190
  Result: SFT WINS
  Crashes: 63
  task-2 ep 1/3 (SFT)...
  task-2 ep 1/3 (heuristic)...
  task-2 ep 2/3 (SFT)...
  task-2 ep 2/3 (heuristic)...
  task-2 ep 3/3 (SFT)...
  task-2 ep 3/3 (heuristic)...

task-2:
  SFT avg reward:       0.6203
  Heuristic avg reward: 0.6092
  Result: SFT WINS
  Crashes: 0
  task-3 ep 1/3 (SFT)...
  task-3 ep 1/3 (heuristic)...
  task-3 ep 2/3 (SFT)...
  task-3 ep 2/3 (heuristic)...
  task-3 ep 3/3 (SFT)...
  task-3 ep 3/3 (heuristic)...

task-3:
  SFT avg reward:       0.6160
  Heuristic avg reward: 0.6067
  Result: SFT WINS
  Crashes: 0

QUALITY GATE SUMMARY
SFT beats heuristic on: 3/3 tasks
Total crashes: 63

Gate results:
  Zero crashes: FAIL
  Beat

In [ ]:
OUTPUT_DIR="/colab/"

In [ ]:
def parse_model_action(text):
    """Extract action from model output text."""
    try:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end < start:
            return None, "no JSON found"
        obj = json.loads(text[start:end+1])
        at = str(obj.get("action_type", "")).upper()
        if at not in VALID_ACTIONS:
            return None, f"invalid action_type: {at}"
        nid = str(obj.get("target_node_id", ""))
        if nid not in VALID_NODES:
            return None, f"invalid target_node_id: {nid}"
        param = float(obj.get("parameter", 0.0))
        return (at, nid, param), "ok"
    except Exception as e:
        return None, str(e)

## 5. REINFORCE Training

This is the core Reinforcement Learning phase. The SFT-trained model is further fine-tuned using the REINFORCE policy gradient algorithm. The model interacts directly with the simulated environment, and its actions are rewarded or penalized, guiding it to learn optimal SRE control policies.

In [9]:
# ============================================================
# Cell 10: REINFORCE Training — Full POC Run (25 episodes)
# All logs, metrics, and graphs saved to /content/POC_logs/
# ============================================================

import torch
import torch.nn.functional as F
import json
import time
import os
from pathlib import Path
from datetime import datetime

# ---- POC Logging Setup ----
POC_DIR = Path("/content/POC_logs")
POC_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = POC_DIR / f"reinforce_train_{TIMESTAMP}.log"
METRICS_FILE = POC_DIR / "reinforce_metrics.json"
PLOT_FILE = POC_DIR / "reinforce_training_curves.png"

# Tee all prints to log file
class TeeLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "w", buffering=1)  # line-buffered
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        self.terminal.flush()
        self.log.flush()

import sys
sys.stdout = TeeLogger(LOG_FILE)
print(f"POC logs directory: {POC_DIR}")
print(f"Training log: {LOG_FILE}")
print(f"Metrics JSON: {METRICS_FILE}")
print(f"Plot output: {PLOT_FILE}")
print()


# -- REINFORCE Config --
class ReinforceConfig:
    num_episodes = 25
    max_steps = 15
    learning_rate = 1e-5
    gamma = 0.99
    baseline_decay = 0.9
    max_grad_norm = 0.1
    temperature = 0.8
    advantage_clip = 1.0
    save_dir = str(OUTPUT_DIR / "reinforce_adapter")
    tasks = ["task-1", "task-2", "task-3"]

rc = ReinforceConfig()

# Switch model back to training mode
model.train()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Optimizer for LoRA parameters only
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=rc.learning_rate)

trainable_count = sum(p.numel() for p in trainable_params)
total_count = sum(p.numel() for p in model.parameters())
print(f"REINFORCE: {trainable_count:,} trainable params ({100*trainable_count/total_count:.2f}%)")
print(f"Config: {rc.num_episodes} episodes, lr={rc.learning_rate}, gamma={rc.gamma}, max_steps={rc.max_steps}")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM before training: {vram_before:.2f}/{vram_total:.2f} GiB")

# Baseline (exponential moving average of returns)
baseline = 0.5
all_episode_returns = []
all_episode_losses = []
all_episode_avg_rewards = []
all_episode_grad_norms = []
all_episode_baselines = []
all_episode_durations = []
all_metrics = []  # Per-episode dict for JSON export

print(f"\n{'='*70}")
print(f"REINFORCE Training — {rc.num_episodes} episodes")
print(f"{'='*70}")

train_start = time.time()

for episode in range(rc.num_episodes):
    ep_start = time.time()
    task_id = rc.tasks[episode % len(rc.tasks)]

    # 1. Collect trajectory (no grad — just generation + env interaction)
    reset_resp = env_reset(task_id=task_id)
    obs_dict = reset_resp.get("observation", reset_resp)
    sla_violations = obs_dict.get("sla_violations", 0)

    step_data = []
    step_rewards = []
    episode_actions = []
    episode_raw_outputs = []  # raw model outputs for debugging

    for step in range(1, rc.max_steps + 1):
        obs_text = format_observation(
            obs_dict, task_id, step, rc.max_steps,
            step_rewards[-1] if step_rewards else 0.0,
            sla_violations
        )

        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{obs_text}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            gen_outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                max_length=None,
                do_sample=True,
                temperature=rc.temperature,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_text = tokenizer.decode(
            gen_outputs[0, prompt_len:],
            skip_special_tokens=True,
        )
        episode_raw_outputs.append(generated_text)

        parsed, err = parse_model_action(generated_text)
        if parsed is None:
            at, nid, param = "NO_OP", "node-0", 0.0
            episode_actions.append(f"INVALID({err})")
        else:
            at, nid, param = parsed
            episode_actions.append(f"{at} {nid} p={param:.2f}")

        step_data.append({
            "full_ids": gen_outputs.clone(),
            "prompt_len": prompt_len,
        })

        step_resp = env_step(at, nid, param)
        obs_dict = step_resp.get("observation", step_resp)
        reward = step_resp.get("reward", 0.0)
        step_rewards.append(reward)
        sla_violations = obs_dict.get("sla_violations", sla_violations)

        if step_resp.get("done", False):
            break

    # 2. Compute discounted returns
    returns = []
    G = 0.0
    for r in reversed(step_rewards):
        G = r + rc.gamma * G
        returns.insert(0, G)

    # 3. Update baseline (EMA)
    episode_return = returns[0] if returns else 0.0
    baseline = rc.baseline_decay * baseline + (1 - rc.baseline_decay) * episode_return

    # 4. Compute REINFORCE loss — forward pass with gradients, one step at a time
    optimizer.zero_grad()
    total_loss_val = 0.0

    for step_idx, sd in enumerate(step_data):
        full_ids = sd["full_ids"].to(DEVICE)
        prompt_len = sd["prompt_len"]

        logits = model(input_ids=full_ids).logits
        gen_logits = logits[0, prompt_len - 1 : -1, :]
        gen_ids = full_ids[0, prompt_len:]

        if gen_ids.numel() == 0 or gen_logits.shape[0] == 0:
            continue

        min_len = min(gen_logits.shape[0], gen_ids.shape[0])
        gen_logits = gen_logits[:min_len]
        gen_ids = gen_ids[:min_len]

        log_probs = F.log_softmax(gen_logits, dim=-1)
        token_log_probs = log_probs.gather(1, gen_ids.unsqueeze(1)).squeeze(1)
        mean_log_prob = token_log_probs.mean()

        advantage = (returns[step_idx] - baseline) if step_idx < len(returns) else 0.0
        advantage = max(-rc.advantage_clip, min(rc.advantage_clip, advantage))
        step_loss = -mean_log_prob * advantage

        step_loss.backward()
        total_loss_val += step_loss.item()

    # 5. Gradient clipping and optimizer step
    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, rc.max_grad_norm).item()
    optimizer.step()

    # Safety: detect NaN after optimizer step
    has_nan = any(
        torch.isnan(p).any().item()
        for p in trainable_params
        if p.grad is not None
    )
    if has_nan:
        print(f"  ⚠ NaN detected after ep {episode+1} — rolling back, resetting optimizer")
        # Reload last saved adapter if available, otherwise skip
        optimizer.zero_grad()
        all_episode_losses.append(float('nan'))
        all_episode_returns.append(episode_return)
        all_episode_avg_rewards.append(sum(step_rewards) / len(step_rewards) if step_rewards else 0.0)
        all_episode_grad_norms.append(grad_norm)
        all_episode_baselines.append(baseline)
        all_episode_durations.append(time.time() - ep_start)
        continue

    # Track metrics
    avg_step_reward = sum(step_rewards) / len(step_rewards) if step_rewards else 0.0
    ep_duration = time.time() - ep_start
    all_episode_returns.append(episode_return)
    all_episode_losses.append(total_loss_val)
    all_episode_avg_rewards.append(avg_step_reward)
    all_episode_grad_norms.append(grad_norm)
    all_episode_baselines.append(baseline)
    all_episode_durations.append(ep_duration)

    # Per-episode metrics dict for JSON
    ep_metrics = {
        "episode": episode + 1,
        "task": task_id,
        "steps": len(step_rewards),
        "avg_reward": round(avg_step_reward, 4),
        "return": round(episode_return, 4),
        "baseline": round(baseline, 4),
        "loss": round(total_loss_val, 4),
        "grad_norm": round(grad_norm, 4),
        "duration_s": round(ep_duration, 1),
        "actions": episode_actions,
    }
    all_metrics.append(ep_metrics)

    # Console logging — always print metrics line
    print(
        f"Ep {episode+1:2d}/{rc.num_episodes} | "
        f"task={task_id} | steps={len(step_rewards):2d} | "
        f"avg_r={avg_step_reward:.3f} | return={episode_return:.3f} | "
        f"baseline={baseline:.3f} | loss={total_loss_val:.3f} | "
        f"grad={grad_norm:.3f} | time={ep_duration:.0f}s"
    )
    # Log actions for first 3, every 5th, and last episode
    if episode < 3 or (episode + 1) % 5 == 0 or episode == rc.num_episodes - 1:
        print(f"  Actions: {', '.join(episode_actions[:20])}")

    # Free GPU memory between episodes
    torch.cuda.empty_cache()

# Save adapter
rc_save = Path(rc.save_dir)
rc_save.mkdir(parents=True, exist_ok=True)
model.save_pretrained(rc.save_dir)
tokenizer.save_pretrained(rc.save_dir)

train_duration = time.time() - train_start

# ---- Summary ----
print(f"\n{'='*70}")
print(f"REINFORCE Training Complete")
print(f"{'='*70}")
print(f"Adapter saved to: {rc.save_dir}")
print(f"Total training time: {train_duration:.0f}s ({train_duration/60:.1f} min)")
if all_episode_returns:
    valid_returns = [r for r in all_episode_returns if r == r]  # filter NaN
    print(f"Episode returns: min={min(valid_returns):.3f}, "
          f"max={max(valid_returns):.3f}, "
          f"mean={sum(valid_returns)/len(valid_returns):.3f}")
if all_episode_losses:
    valid_losses = [l for l in all_episode_losses if l == l]
    print(f"Episode losses:  min={min(valid_losses):.3f}, max={max(valid_losses):.3f}")
if torch.cuda.is_available():
    vram_after = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM: {vram_after:.2f}/{vram_total:.2f} GiB")

# ---- Save Metrics JSON ----
with open(METRICS_FILE, "w") as f:
    json.dump({
        "config": {
            "num_episodes": rc.num_episodes,
            "max_steps": rc.max_steps,
            "learning_rate": rc.learning_rate,
            "gamma": rc.gamma,
            "baseline_decay": rc.baseline_decay,
            "max_grad_norm": rc.max_grad_norm,
            "temperature": rc.temperature,
            "advantage_clip": rc.advantage_clip,
            "tasks": rc.tasks,
        },
        "summary": {
            "total_time_s": round(train_duration, 1),
            "mean_return": round(sum(valid_returns) / len(valid_returns), 4) if valid_returns else None,
            "min_return": round(min(valid_returns), 4) if valid_returns else None,
            "max_return": round(max(valid_returns), 4) if valid_returns else None,
        },
        "episodes": all_metrics,
        "timeseries": {
            "returns": [round(r, 4) for r in all_episode_returns],
            "avg_rewards": [round(r, 4) for r in all_episode_avg_rewards],
            "losses": [round(l, 4) if l == l else None for l in all_episode_losses],
            "grad_norms": [round(g, 4) for g in all_episode_grad_norms],
            "baselines": [round(b, 4) for b in all_episode_baselines],
            "durations": [round(d, 1) for d in all_episode_durations],
        },
    }, f, indent=2)
print(f"Metrics saved to: {METRICS_FILE}")

# ---- Generate Training Curves Plot ----
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("REINFORCE Training Curves", fontsize=14, fontweight="bold")

    eps = list(range(1, len(all_episode_returns) + 1))

    # 1. Episode Returns + Baseline
    ax = axes[0, 0]
    ax.plot(eps, all_episode_returns, "b-o", markersize=3, label="Episode Return")
    ax.plot(eps, all_episode_baselines, "r--", linewidth=1.5, label="Baseline (EMA)")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Return")
    ax.set_title("Returns vs Baseline")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Average Step Reward
    ax = axes[0, 1]
    ax.plot(eps, all_episode_avg_rewards, "g-o", markersize=3)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Avg Step Reward")
    ax.set_title("Average Reward per Episode")
    ax.grid(True, alpha=0.3)

    # 3. Loss
    ax = axes[1, 0]
    valid_eps = [e for e, l in zip(eps, all_episode_losses) if l == l]
    valid_loss = [l for l in all_episode_losses if l == l]
    ax.plot(valid_eps, valid_loss, "r-o", markersize=3)
    ax.axhline(y=0, color="k", linestyle="--", alpha=0.3)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Loss")
    ax.set_title("REINFORCE Loss")
    ax.grid(True, alpha=0.3)

    # 4. Gradient Norm
    ax = axes[1, 1]
    ax.plot(eps, all_episode_grad_norms, "m-o", markersize=3)
    ax.axhline(y=rc.max_grad_norm, color="r", linestyle="--", alpha=0.5, label=f"clip={rc.max_grad_norm}")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Grad Norm (pre-clip)")
    ax.set_title("Gradient Norm")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOT_FILE, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved to: {PLOT_FILE}")
except Exception as e:
    print(f"Plot failed: {e}")

# ---- Copy adapter to POC logs ----
import shutil
poc_adapter = POC_DIR / "reinforce_adapter"
if poc_adapter.exists():
    shutil.rmtree(poc_adapter)
shutil.copytree(rc.save_dir, poc_adapter)
print(f"Adapter copied to: {poc_adapter}")

print(f"\nAll POC artifacts in: {POC_DIR}")
print(f"  - {LOG_FILE.name}  (full training log)")
print(f"  - {METRICS_FILE.name}  (structured metrics)")
print(f"  - {PLOT_FILE.name}  (training curves)")
print(f"  - reinforce_adapter/  (LoRA adapter)")


POC logs directory: /content/POC_logs
Training log: /content/POC_logs/reinforce_train_20260426_082401.log
Metrics JSON: /content/POC_logs/reinforce_metrics.json
Plot output: /content/POC_logs/reinforce_training_curves.png

REINFORCE: 3,686,400 trainable params (0.22%)
Config: 25 episodes, lr=1e-05, gamma=0.99, max_steps=15
Device: cuda
VRAM before training: 2.21/15.64 GiB

REINFORCE Training — 25 episodes
Unsloth: Will smartly offload gradients to save VRAM!
Ep  1/25 | task=task-1 | steps=15 | avg_r=0.620 | return=8.673 | baseline=1.317 | loss=5.512 | grad=8.610 | time=87s
  Actions: SCALE_UP node-0 p=3.00, SCALE_UP node-0 p=2.50, SCALE_UP node-4 p=0.25, NO_OP node-0 p=0.00, SCALE_UP node-0 p=1.50, SCALE_UP node-0 p=3.00, NO_OP node-0 p=0.00, SCALE_DOWN node-4 p=0.50, REROUTE_TRAFFIC node-4 p=0.70, REROUTE_TRAFFIC node-4 p=0.60, INVALID(no JSON found), INVALID(no JSON found), INVALID(Extra data: line 1 column 82 (char 81)), REROUTE_TRAFFIC node-4 p=0.75, REROUTE_TRAFFIC node-4 p=0.20
E